# Sesión 4: FILM | Entre imágenes

**Arqueología de la I.A. generativa · Taller MUTAR**

**Año de referencia:** 2022 | **Modelo:** FILM, *Frame Interpolation for Large Motion* (Google Research y Universidad de Washington)

---

### Antes de empezar

1. **Haz tu propia copia:** `Archivo → Guardar una copia en Drive`.
2. **Activa la GPU:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar`.
3. **Ejecuta las celdas en orden**, de arriba hacia abajo (botón ▶ o `Ctrl + Enter`).
4. Las celdas con **formulario** se modifican sin tocar el código: cambia el valor a la derecha y vuelve a ejecutar solo esa celda.

**No necesitas saber programar.**

### Qué haremos hoy

Esta es la sesión que junta todo. Vamos a tomar lo que hiciste en las sesiones anteriores (sueños de DeepDream, criaturas de BigGAN, habitaciones y Pokémon de Projected GAN) y conectarlo en piezas de video:

1. **Secuencias:** de una imagen o un video a otro, con FILM inventando todo lo que pasa en el medio. Puedes encadenar videos de distintas sesiones en una sola pieza.
2. **Zoom infinito:** la cámara entra en una imagen (o un video) y cae en la siguiente, sin cortes.
3. **Cámara lenta:** el uso original de FILM, inventar frames entre los frames de un video.

**Te llevas:** tus piezas finales del taller.

## ¿Qué es FILM?

**FILM** (Fitsum Reda, Janne Kontkanen, Eric Tabellion, Deqing Sun, Caroline Pantofaru y Brian Curless, 2022) es una red de **interpolación de frames**: recibe dos imágenes y genera las que irían entre medio.

No es un fundido (que solo superpone una imagen sobre la otra, con transparencia). FILM **estima el movimiento**: mira qué partes de la primera imagen corresponden a qué partes de la segunda, las desplaza a mitad de camino y las funde en una imagen nueva.

* **Para qué se hizo:** para fotos casi iguales, como dos fotos de una ráfaga, y crear un video en cámara lenta entre ellas. Su especialidad son los **movimientos grandes** (*large motion*), donde otros interpoladores fallaban.
* **Con qué aprendió:** con miles de tríos de frames seguidos de videos reales (el primero, el del medio y el último). Su tarea era adivinar el del medio.

> **Lo que hacemos hoy está fuera de su entrenamiento.** Entre una habitación de Projected GAN y un sueño de DeepDream no hay "movimiento" real. FILM igual busca correspondencias donde no las hay: deforma, estira y funde. No es un error a evitar: es la estética de esta sesión.

### Las perillas de hoy

* **Interpolaciones:** cuántas veces FILM parte en dos el camino entre dos imágenes. Cada vez se duplican los frames.
* **Formato y tamaño:** cuadrado, horizontal o vertical; 512, 768 o 1024 píxeles.
* **Zoom:** cuánto entra la cámara en cada imagen.

In [ ]:
#@title 0. Preparar el entorno (ejecutar primero)
import os, sys, subprocess, time, glob, shutil, json, math
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow-hub", "imageio-ffmpeg"], check=True)

import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageOps
from IPython.display import display, Video
from tqdm.auto import tqdm
from google.colab import files, drive

if tf.config.list_physical_devices("GPU"):
    print("GPU activada.")
else:
    print("ATENCIÓN: no hay GPU activa. FILM funciona, pero mucho más lento.")
    print("Ve a Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar,")
    print("y vuelve a ejecutar esta celda.")

CARPETA = "/content/resultados/sesion4"
SUBIDAS = "/content/subidas"
os.makedirs(CARPETA, exist_ok=True)
os.makedirs(SUBIDAS, exist_ok=True)
CARPETA_TALLER = None

### 0.1 Conectar tu Google Drive (recomendado)

Conectar Drive sirve para dos cosas:

1. **Usar lo que hiciste en las sesiones anteriores** directamente desde tus carpetas, sin descargar ni subir nada.
2. **Guardar** lo que hagas hoy en `Taller_MUTAR/sesion4`.

```
Mi unidad/
└── Taller_MUTAR/
    ├── sesion1/   ← DeepDream
    ├── sesion2/   ← BigGAN
    ├── sesion3/   ← Projected GAN
    └── sesion4/   ← hoy
```

Si prefieres no conectarlo, desmarca la casilla: podrás subir imágenes y videos desde tu computador (sección 3) y descargar todo al final.

In [ ]:
#@title 0.1 Conectar Drive
USAR_DRIVE = True #@param {type:"boolean"}
RUTA_TALLER = "/content/drive/MyDrive/Taller_MUTAR" #@param {type:"string"}

if USAR_DRIVE:
    try:
        drive.mount("/content/drive")
        CARPETA_TALLER = RUTA_TALLER
        CARPETA = os.path.join(CARPETA_TALLER, "sesion4")
        os.makedirs(CARPETA, exist_ok=True)
    except Exception as e:
        CARPETA_TALLER = None
        print("No se pudo conectar Drive (permite las ventanas emergentes y usa la misma cuenta de Google que en Colab).")
        print("Puedes seguir subiendo imágenes desde tu computador:", e)
print("Los resultados se guardarán en:", CARPETA)

## 1. Cargar FILM

Descarga el modelo ya entrenado desde TensorFlow Hub. La primera vez tarda uno o dos minutos.

In [ ]:
#@title 1. Cargar FILM
print("Cargando FILM...")
modelo = hub.load("https://tfhub.dev/google/film/1")
print("FILM listo.")

## 2. Formato de tus piezas

Todas las imágenes y videos se llevan al mismo formato antes de interpolar.

* **FORMATO:** cuadrado (como casi todo lo del taller), horizontal 16:9 (pantalla, proyección) o vertical 9:16 (celular, historias).
* **TAMAÑO:** el lado más largo, en píxeles. Más grande = más detalle, pero FILM tarda más (1024 tarda unas cuatro veces más que 512).
* **AJUSTE**, cuando la imagen no tiene la forma del formato:
  * **recortar:** llena todo el cuadro y corta lo que sobra.
  * **encajar:** muestra la imagen completa y rellena los bordes con el **FONDO** que elijas (*difuminado* usa la misma imagen, agrandada y borrosa).

**FPS** y **CALIDAD** son para los videos, como en las otras sesiones. Si cambias algo aquí, vuelve a ejecutar la celda.

In [ ]:
#@title 2. Formato de imagen y video
FORMATO = "cuadrado" #@param ["cuadrado", "horizontal 16:9", "vertical 9:16"]
TAMANO = 512 #@param [512, 768, 1024] {type:"raw"}
AJUSTE = "recortar" #@param ["recortar", "encajar"]
FONDO = "difuminado" #@param ["difuminado", "negro", "blanco"]
FPS = 24 #@param [12, 24, 25, 30] {type:"raw"}
#@markdown CALIDAD: 18 = muy alta (archivo grande), 23 = buena, 28 = liviana (para WhatsApp)
CALIDAD = 18 #@param {type:"slider", min:14, max:30, step:1}

def medidas():
    if FORMATO == "cuadrado":
        return TAMANO, TAMANO
    corto = int(round(TAMANO * 9 / 16 / 2)) * 2   # siempre par, para el video
    return (TAMANO, corto) if FORMATO.startswith("horizontal") else (corto, TAMANO)

ANCHO, ALTO = medidas()
print(f"Formato: {FORMATO} · {ANCHO} x {ALTO} px · {AJUSTE}" + (f" (fondo {FONDO})" if AJUSTE == "encajar" else "") + f" · {FPS} FPS")

### 2.1 El motor (solo ejecutar)

Define las funciones que preparan las imágenes, llaman a FILM y arman los videos. No hace falta modificarla.

In [ ]:
#@title 2.1 Motor (solo ejecutar)

# --- Avisos: si algo está mal escrito, se muestra solo el mensaje, sin el error largo de Python ---
import re
class Aviso(Exception):
    pass
def _mostrar_aviso(shell, etype, evalue, tb, tb_offset=None):
    print("AVISO:", evalue)
try:
    get_ipython().set_custom_exc((Aviso,), _mostrar_aviso)
except NameError:
    pass

# --- Preparar imágenes al formato ---
def ajustar(img, escala=1):
    # Lleva cualquier imagen (PIL) al formato elegido en la celda 2 (escala > 1: más grande, para el zoom).
    ancho, alto = int(ANCHO * escala), int(ALTO * escala)
    img = ImageOps.exif_transpose(img).convert("RGB")
    if AJUSTE == "recortar":
        return ImageOps.fit(img, (ancho, alto), Image.LANCZOS)
    if FONDO == "difuminado":
        fondo = ImageOps.fit(img, (ancho, alto), Image.LANCZOS).filter(ImageFilter.GaussianBlur(ancho / 25))
    else:
        fondo = Image.new("RGB", (ancho, alto), (0, 0, 0) if FONDO == "negro" else (255, 255, 255))
    figura = ImageOps.contain(img, (ancho, alto), Image.LANCZOS)
    fondo.paste(figura, ((ancho - figura.width) // 2, (alto - figura.height) // 2))
    return fondo

def a_arreglo(img):
    return np.asarray(img, dtype=np.float32) / 255.0

def a_imagen(arr):
    return Image.fromarray((np.clip(arr, 0, 1) * 255 + 0.5).astype(np.uint8))

def cargar(ruta):
    return a_arreglo(ajustar(Image.open(ruta)))

# --- FILM ---
def interpolar(x0, x1, t=0.5):
    # Un frame entre x0 y x1. t = 0 es x0, t = 1 es x1, t = 0.5 es la mitad.
    # FILM trabaja mejor con medidas divisibles por 64: rellenamos y después recortamos.
    h, w = x0.shape[:2]
    H, W = math.ceil(h / 64) * 64, math.ceil(w / 64) * 64
    arriba, izq = (H - h) // 2, (W - w) // 2
    relleno = ((arriba, H - h - arriba), (izq, W - w - izq), (0, 0))
    entrada = {"x0": np.pad(x0, relleno)[None], "x1": np.pad(x1, relleno)[None],
               "time": np.array([[t]], dtype=np.float32)}
    salida = modelo(entrada, training=False)["image"][0].numpy()
    return np.clip(salida[arriba:arriba + h, izq:izq + w], 0, 1)

def tramo(a, b, veces, escribir):
    # Escribe a y los frames intermedios entre a y b (no escribe b).
    # Parte el camino a la mitad, y cada mitad otra vez a la mitad, "veces" veces.
    if veces == 0:
        escribir(a)
        return
    medio = interpolar(a, b, 0.5)
    tramo(a, medio, veces - 1, escribir)
    tramo(medio, b, veces - 1, escribir)

# --- Frames y videos ---
class Escritor:
    # Guarda cada frame en disco apenas sale (no se llena la memoria) y anota el orden.
    def __init__(self, nombre):
        self.nombre = nombre
        self.carpeta = os.path.join(CARPETA, f"frames_{nombre}")
        shutil.rmtree(self.carpeta, ignore_errors=True)
        os.makedirs(self.carpeta)
        self.orden, self.n = [], 0
    def __call__(self, arr, repetir=1):
        archivo = f"{self.n:06d}.jpg"
        a_imagen(arr).save(os.path.join(self.carpeta, archivo), quality=95)
        self.n += 1
        self.orden += [archivo] * repetir
    def cerrar(self):
        with open(os.path.join(self.carpeta, "orden.json"), "w") as f:
            json.dump(self.orden, f)

def ffmpeg():
    return shutil.which("ffmpeg") or __import__("imageio_ffmpeg").get_ffmpeg_exe()

def montar_video(nombre, fps=None, ida_y_vuelta=False, al_reves=False, calidad=None, gif=False, mostrar=True):
    # Une los frames de la carpeta en un .mp4 (y opcionalmente un .gif liviano).
    fps, calidad = fps or FPS, calidad or CALIDAD
    carpeta = os.path.join(CARPETA, f"frames_{nombre}")
    ruta_orden = os.path.join(carpeta, "orden.json")
    if os.path.exists(ruta_orden):
        frames = [os.path.join(carpeta, f) for f in json.load(open(ruta_orden))]
    else:
        frames = sorted(glob.glob(os.path.join(carpeta, "*.jpg")))
    if not frames:
        raise Aviso(f"No hay frames para '{nombre}'.")
    if al_reves:
        frames = frames[::-1]
    if ida_y_vuelta:
        frames = frames + frames[-2:0:-1]
    tmp = "/content/_montaje"
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    for i, f in enumerate(frames):
        os.symlink(f, os.path.join(tmp, f"{i:06d}.jpg"))
    salida = os.path.join(CARPETA, f"{nombre}.mp4")
    subprocess.run([ffmpeg(), "-y", "-loglevel", "error", "-framerate", str(fps), "-i", f"{tmp}/%06d.jpg",
                    "-c:v", "libx264", "-preset", "medium", "-crf", str(calidad),
                    "-pix_fmt", "yuv420p", "-movflags", "+faststart", salida], check=True)
    mb = os.path.getsize(salida) / 1e6
    print(f"Video: {salida} · {len(frames)} frames · {len(frames) / fps:.1f} s · {mb:.1f} MB")
    if gif:
        salida_gif = os.path.join(CARPETA, f"{nombre}.gif")
        subprocess.run([ffmpeg(), "-y", "-loglevel", "error", "-i", salida, "-vf",
                        "fps=12,scale=384:-2:flags=lanczos,split[a][b];[a]palettegen[p];[b][p]paletteuse",
                        salida_gif], check=True)
        print(f"GIF: {salida_gif} · {os.path.getsize(salida_gif) / 1e6:.1f} MB")
    if mostrar and mb < 40:
        display(Video(salida, embed=True, width=min(ANCHO, 640)))
    elif mostrar:
        print("(El video es muy pesado para mostrarlo aquí: descárgalo en la sección 7 o búscalo en tu Drive.)")
    return salida

def leer_video(ruta, fps=None, desde=0.0, hasta=0.0):
    # Extrae los frames de un video (ya ajustados al formato) a una carpeta temporal.
    tmp = f"/content/_video_{time.time_ns()}"   # carpeta nueva cada vez
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    cmd = [ffmpeg(), "-y", "-loglevel", "error"]
    if desde > 0:
        cmd += ["-ss", str(desde)]
    cmd += ["-i", ruta]
    if hasta > desde:
        cmd += ["-t", str(hasta - desde)]
    if fps:
        cmd += ["-vf", f"fps={fps}"]
    subprocess.run(cmd + ["-q:v", "2", f"{tmp}/%06d.jpg"], check=True)
    return sorted(glob.glob(f"{tmp}/*.jpg"))

def fps_de(ruta):
    # FPS original de un video.
    salida = subprocess.run([ffmpeg(), "-i", ruta], capture_output=True, text=True).stderr
    for parte in salida.split(","):
        if parte.strip().endswith(" fps"):
            return float(parte.strip().split()[0])
    return 24.0

# --- Zoom ---
def acercar(img, factor):
    # Recorta el centro y lo lleva al tamaño del formato: la cámara se acerca "factor" veces.
    # img puede estar guardada más grande que el formato, para que el zoom no se vea pixelado.
    w, h = img.size
    cw, ch = w / factor, h / factor
    caja = ((w - cw) / 2, (h - ch) / 2, (w + cw) / 2, (h + ch) / 2)
    return img.resize((ANCHO, ALTO), Image.LANCZOS, box=caja)

def incrustar(base, img, factor):
    # Pega img achicada (factor < 1) en el centro de base.
    w, h = base.size
    chica = img.resize((max(1, round(w * factor)), max(1, round(h * factor))), Image.LANCZOS)
    # Borde suave: mientras la imagen es chica, sus bordes se desvanecen; al llenar el cuadro, ya no.
    r = int(min(chica.size) * 0.12 * (1 - factor))
    mascara = Image.new("L", chica.size, 255)
    if r > 0:
        mascara = Image.new("L", chica.size, 0)
        ImageDraw.Draw(mascara).rectangle((r, r, chica.width - r, chica.height - r), fill=255)
        mascara = mascara.filter(ImageFilter.GaussianBlur(r / 2))
    copia = base.copy()
    copia.paste(chica, ((w - chica.width) // 2, (h - chica.height) // 2), mascara)
    return copia

# --- Utilidades ---
def fuente(tam=14):
    try:
        from matplotlib import font_manager
        return ImageFont.truetype(font_manager.findfont("DejaVu Sans"), tam)
    except Exception:
        return ImageFont.load_default()

def hoja(imagenes, etiquetas, columnas=6, lado=160):
    filas = (len(imagenes) + columnas - 1) // columnas
    h = Image.new("RGB", (columnas * lado, filas * (lado + 22)), "white")
    d, f = ImageDraw.Draw(h), fuente()
    for k, im in enumerate(imagenes):
        x, y = (k % columnas) * lado, (k // columnas) * (lado + 22)
        im = ImageOps.contain(im.convert("RGB"), (lado - 4, lado - 4))
        h.paste(im, (x + (lado - im.width) // 2, y + (lado - im.height) // 2))
        d.text((x + 4, y + lado + 2), etiquetas[k], fill=(0, 0, 0), font=f)
    return h

def limpio(texto):
    return "".join(c if c.isalnum() else "_" for c in str(texto))

# --- Catálogo ---
EXT_IMG = (".png", ".jpg", ".jpeg", ".webp")
EXT_VID = (".mp4", ".mov", ".webm")

def es_video(ruta):
    return ruta.lower().endswith(EXT_VID)

def corto(ruta):
    base = CARPETA_TALLER + "/" if CARPETA_TALLER and ruta.startswith(CARPETA_TALLER) else "/content/"
    return ruta.replace(base, "")

MINIS = {}
def miniatura(ruta, lado=156):
    # Miniatura de una imagen, o del primer frame de un video. Se guarda para no volver a leerla.
    clave = (ruta, os.path.getmtime(ruta))
    if clave not in MINIS:
        origen = ruta
        if es_video(ruta):
            origen = f"/content/_mini_{len(MINIS)}.jpg"
            subprocess.run([ffmpeg(), "-y", "-loglevel", "error", "-i", ruta, "-frames:v", "1", "-q:v", "3", origen])
        try:
            with Image.open(origen) as im:
                im.draft("RGB", (lado * 2, lado * 2))   # en JPG, lee directo en tamaño chico
                im = im.convert("RGB")
                im.thumbnail((lado, lado))
        except Exception:
            im = Image.new("RGB", (lado, lado), (220, 220, 220))
        MINIS[clave] = im
    return MINIS[clave]

def elegir(texto):
    # Convierte "3, 1, 7" en la lista de archivos correspondientes del catálogo.
    if "MATERIAL" not in globals():
        raise Aviso("Primero ejecuta la celda 3 (tu material).")
    elegidos = []
    for parte in re.split(r"[\s,;]+", str(texto).strip()):   # acepta "4, 12, 7" o "4 12 7"
        if not parte:
            continue
        if not parte.isdigit() or not 1 <= int(parte) <= len(MATERIAL):
            raise Aviso(f"'{parte}' no es un número del catálogo (sección 3).")
        elegidos.append(MATERIAL[int(parte) - 1])
    return elegidos

class Fuente:
    # Una imagen o un video del catálogo, ya llevado al formato de la celda 2.
    # imagen(k) da el frame k (una imagen siempre da lo mismo; un video vuelve a empezar si se acaba).
    def __init__(self, ruta, escala=1, max_seg=0.0):
        self.es_video = es_video(ruta)
        self.escala = escala
        self._memoria = {}
        if self.es_video:
            self.frames = leer_video(ruta, fps=FPS, hasta=max_seg)
            if not self.frames:
                raise Aviso(f"No se pudieron leer frames de {corto(ruta)}.")
        else:
            self.fija = ajustar(Image.open(ruta), escala)
    def __len__(self):
        return len(self.frames) if self.es_video else 1
    def imagen(self, k):
        if not self.es_video:
            return self.fija
        k %= len(self.frames)
        if k not in self._memoria:
            if len(self._memoria) > 8:
                self._memoria.clear()
            self._memoria[k] = ajustar(Image.open(self.frames[k]), self.escala)
        return self._memoria[k]
    def arreglo(self, k):
        return a_arreglo(self.imagen(k))

def limpiar_temporales():
    for d in glob.glob("/content/_video_*"):
        shutil.rmtree(d, ignore_errors=True)

print("Motor listo.")

## 3. Tu material

Esta celda arma un **catálogo numerado** con tus imágenes y videos, todos en una sola lista. Elige el **ORIGEN** y ejecútala: termina sola cuando acaba de subir o de recorrer las carpetas.

* **Drive (carpetas del taller):** recorre `Taller_MUTAR/sesion1` a `sesion4` (necesitas la celda 0.1 conectada), sin las carpetas `frames_…`. Con **CARPETAS** puedes mirar una sola sesión: si tienes mucho material, es más rápido.
* **Mi computador:** aparece el botón **«Elegir archivos»** (imágenes PNG o JPG, videos MP4). Cuando terminan de subir, la celda termina.

**Para combinar las dos cosas:** primero sube desde tu computador y después ejecuta la celda con Drive. Lo que subiste se suma al final del catálogo.

En las miniaturas, los videos aparecen con su primer frame y la palabra **video** junto al número. La primera vez pueden tardar un poco (se leen desde Drive); la barra muestra cuánto falta. Después salen al instante. Si solo quieres la lista, desmarca **MOSTRAR_MINIATURAS**.

En las secciones siguientes eliges el material **escribiendo sus números** en el orden que quieras.

In [ ]:
#@title 3. Ver mi material (catálogo)
ORIGEN = "Drive (carpetas del taller)" #@param ["Drive (carpetas del taller)", "Mi computador"]
CARPETAS = "todas" #@param ["todas", "sesion1", "sesion2", "sesion3", "sesion4"]
MOSTRAR_MINIATURAS = True #@param {type:"boolean"}

# 1. Subir desde el computador
if ORIGEN == "Mi computador":
    print("Haz clic en «Elegir archivos» y selecciona tus imágenes o videos (puedes elegir varios a la vez).")
    subidos = files.upload()
    for nombre, datos in subidos.items():
        with open(os.path.join(SUBIDAS, nombre), "wb") as f:
            f.write(datos)
        if os.path.exists(nombre) and os.path.abspath(nombre) != os.path.join(SUBIDAS, nombre):
            os.remove(nombre)
    print(f"Subidos: {len(subidos)} archivos.")

# 2. Buscar en Drive y en lo subido
fuentes = []
if ORIGEN.startswith("Drive"):
    if CARPETA_TALLER and os.path.isdir(CARPETA_TALLER):
        patron = "sesion*" if CARPETAS == "todas" else CARPETAS
        fuentes += sorted(d for d in glob.glob(os.path.join(CARPETA_TALLER, patron)) if os.path.isdir(d))
    else:
        print("Drive no está conectado (celda 0.1): elige «Mi computador» para subir tus archivos.")
fuentes.append(SUBIDAS)   # lo que ya subiste desde el computador siempre aparece

MATERIAL = []
for carpeta in fuentes:
    antes = len(MATERIAL)
    for raiz, dirs, archivos in os.walk(carpeta):
        dirs[:] = sorted(d for d in dirs if not d.startswith(("frames_", "_")))   # sin carpetas de frames
        MATERIAL += [os.path.join(raiz, a) for a in sorted(archivos) if a.lower().endswith(EXT_IMG + EXT_VID)]
    print(f"Revisado {os.path.basename(carpeta)}: {len(MATERIAL) - antes} archivos")

n_vid = sum(es_video(r) for r in MATERIAL)
print(f"\nMATERIAL: {len(MATERIAL) - n_vid} imágenes y {n_vid} videos")
for i, r in enumerate(MATERIAL, 1):
    print(f"  {i:3d}  {'video ' if es_video(r) else '      '} {corto(r)}")

# 3. Miniaturas (se guardan para que la próxima vez salgan al instante)
if MOSTRAR_MINIATURAS and MATERIAL:
    minis = [miniatura(r) for r in tqdm(MATERIAL, desc="Miniaturas")]
    display(hoja(minis, [f"{i} video" if es_video(r) else str(i) for i, r in enumerate(MATERIAL, 1)]))

## 4. Secuencia de imágenes y videos

Escribe en **ELEGIDOS** los números del catálogo, en el orden en que quieres que aparezcan (por ejemplo `4, 12, 7`). Puedes mezclar imágenes y videos. Entre cada uno y el siguiente, FILM genera la transición.

* Una **imagen** aparece y se transforma en lo que sigue.
* Un **video** se reproduce y, desde su último frame, se transforma en lo que sigue. Así puedes encadenar videos de distintas sesiones en una sola pieza.

**INTERPOLACIONES:** cuántas veces FILM parte el camino a la mitad. Cada vez se duplican los frames de la transición:

| Interpolaciones | Frames de transición | A 24 FPS dura |
|---|---|---|
| 3 | 8 | 0,3 s |
| 5 | 32 | 1,3 s |
| 6 | 64 | 2,7 s |
| 7 | 128 | 5,3 s |

Para transiciones más lentas sin más trabajo para FILM, baja los FPS en la celda 2.

* **PAUSA_SEG (opcional):** cuántos segundos se queda quieta cada **imagen** antes de transformarse. Con 0, la transformación no se detiene nunca.
* **SEGUNDOS_DE_VIDEO (opcional):** usar solo los primeros segundos de cada video. Con 0, se usa completo.
* **VOLVER_AL_INICIO:** agrega la transición de vuelta al primero, para un bucle perfecto.

In [ ]:
#@title 4. Secuencia de imágenes y videos
NOMBRE = "secuencia" #@param {type:"string"}
ELEGIDOS = "1, 2, 3" #@param {type:"string"}
INTERPOLACIONES = 5 #@param {type:"slider", min:1, max:7, step:1}
PAUSA_SEG = 0 #@param {type:"slider", min:0, max:5, step:0.5}
SEGUNDOS_DE_VIDEO = 0 #@param {type:"number"}
VOLVER_AL_INICIO = False #@param {type:"boolean"}
HACER_GIF = False #@param {type:"boolean"}

piezas = [Fuente(r, max_seg=float(SEGUNDOS_DE_VIDEO)) for r in elegir(ELEGIDOS)]
if len(piezas) < 2:
    raise Aviso("Elige al menos dos.")
if VOLVER_AL_INICIO:
    piezas.append(piezas[0])
display(hoja([p.imagen(0) for p in piezas], [f"{i + 1}" + (" video" if p.es_video else "") for i, p in enumerate(piezas)],
             columnas=min(len(piezas), 6)))

pausa = max(1, int(round(PAUSA_SEG * FPS)))   # cuántas veces se repite cada imagen
escribir = Escritor(limpio(NOMBRE))
for i, p in enumerate(tqdm(piezas, desc="Secuencia")):
    ultima = i == len(piezas) - 1
    # 1. La pieza: una imagen (con su pausa) o el video completo
    if p.es_video and not (ultima and VOLVER_AL_INICIO):
        for k in range(len(p)):
            escribir(p.arreglo(k))
    else:
        escribir(p.arreglo(0), repetir=1 if (ultima and VOLVER_AL_INICIO) else pausa)
    # 2. La transición hacia la siguiente (sin repetir el frame de partida, que ya está)
    if not ultima:
        saltar = [True]
        def escribir_transicion(x):
            if saltar[0]:
                saltar[0] = False
            else:
                escribir(x)
        tramo(p.arreglo(len(p) - 1), piezas[i + 1].arreglo(0), INTERPOLACIONES, escribir_transicion)
escribir.cerrar()
limpiar_temporales()
montar_video(limpio(NOMBRE), gif=HACER_GIF)

## 5. Zoom infinito

La cámara **entra** en una imagen y, sin cortes, cae en la siguiente.

Cómo funciona: la cámara se acerca a la imagen actual. En el centro, FILM transforma poco a poco lo que se ve ahí en la imagen siguiente, mientras esa zona crece. Al empezar, el centro es igual al resto de la imagen (no se nota ningún recorte); al final del tramo, la imagen siguiente ocupa todo el cuadro y el viaje sigue hacia adentro de ella.

**También funciona con videos:** el video se sigue moviendo mientras la cámara cae dentro de él. Si es más corto que el tramo, vuelve a empezar.

* **ZOOM:** cuánto se acerca la cámara en cada tramo. 2 = el doble. Valores altos dan una caída más vertiginosa. Si tus imágenes son chicas (por ejemplo, de 256 px), con zoom alto se ven pixeladas antes de cambiar.
* **SEGUNDOS_POR_TRAMO:** cuánto dura cada caída.
* **DIRECCION:** *adentro* (caer) o *afuera* (salir: el mismo viaje al revés; los videos también corren al revés).

In [ ]:
#@title 5. Zoom infinito
NOMBRE_ZOOM = "zoom_infinito" #@param {type:"string"}
ELEGIDOS_ZOOM = "1, 2, 3" #@param {type:"string"}
ZOOM = 2.5 #@param {type:"slider", min:1.5, max:6, step:0.5}
SEGUNDOS_POR_TRAMO = 4 #@param {type:"slider", min:1, max:20, step:1}
DIRECCION = "adentro" #@param ["adentro", "afuera"]
VOLVER_AL_INICIO_Z = False #@param {type:"boolean"}
HACER_GIF_Z = False #@param {type:"boolean"}

escala = min(ZOOM, 3)   # guardamos las imágenes más grandes para que el zoom no se pixele tanto
piezas = [Fuente(r, escala=escala) for r in elegir(ELEGIDOS_ZOOM)]
if len(piezas) < 2:
    raise Aviso("Elige al menos dos.")
if VOLVER_AL_INICIO_Z:
    piezas.append(piezas[0])

n = max(2, int(round(SEGUNDOS_POR_TRAMO * FPS)))
pasos = min(8, max(3, math.ceil(math.log2(n))))   # FILM parte el camino a la mitad "pasos" veces
print(f"{len(piezas) - 1} caídas · {n} frames cada una · {(len(piezas) - 1) * n / FPS:.1f} s · zoom x{ZOOM} por tramo")
escribir = Escritor(limpio(NOMBRE_ZOOM))
for i in tqdm(range(len(piezas) - 1), desc="Caídas"):
    a, b = piezas[i], piezas[i + 1]
    # 1. FILM transforma el centro de a (lo que tapa b al empezar) en b, siempre partiendo a la mitad.
    centro_a = a_arreglo(acercar(a.imagen(0), ZOOM))
    destino_b = a_arreglo(acercar(b.imagen(0), 1))
    metamorfosis = []
    tramo(centro_a, destino_b, pasos, metamorfosis.append)
    metamorfosis.append(destino_b)
    # 2. La cámara entra en a; en el centro crece la metamorfosis hasta llenar el cuadro.
    for k in range(n):
        t = k / n
        pos = t * (len(metamorfosis) - 1)
        j, resto = int(pos), pos - int(pos)
        centro = metamorfosis[j] if resto == 0 else (1 - resto) * metamorfosis[j] + resto * metamorfosis[j + 1]
        vista_a = acercar(a.imagen(k), ZOOM ** t)                          # la cámara entra en a
        escribir(a_arreglo(incrustar(vista_a, a_imagen(centro), ZOOM ** (t - 1))))
escribir(a_arreglo(acercar(piezas[-1].imagen(0), 1)))
escribir.cerrar()
limpiar_temporales()
montar_video(limpio(NOMBRE_ZOOM), al_reves=(DIRECCION == "afuera"), gif=HACER_GIF_Z)

## 6. Cámara lenta

El uso original de FILM: inventar frames **entre los frames** de un video.

* **VIDEO:** el número del catálogo (tiene que ser un video).
* **MULTIPLICAR:** cuántos frames hay después por cada frame original (x2, x4, x8).
* **RESULTADO:**
  * *más lento:* el video dura MULTIPLICAR veces más (cámara lenta).
  * *más fluido:* dura lo mismo, pero con MULTIPLICAR veces más frames por segundo.
* **DESDE_SEG / HASTA_SEG:** el pedazo del video a procesar. Empieza con pocos segundos: cada frame nuevo es una llamada a FILM.

Funciona muy bien con los videos de DeepDream (Sesión 1). Con los de las sesiones 2 y 3, que ya son muy suaves, se nota menos.

In [ ]:
#@title 6. Cámara lenta
NOMBRE_LENTO = "camara_lenta" #@param {type:"string"}
VIDEO = 1 #@param {type:"integer"}
MULTIPLICAR = 2 #@param [2, 4, 8] {type:"raw"}
RESULTADO = "más lento" #@param ["más lento", "más fluido"]
DESDE_SEG = 0 #@param {type:"number"}
HASTA_SEG = 5 #@param {type:"number"}
HACER_GIF_L = False #@param {type:"boolean"}

ruta = elegir(str(VIDEO))[0]
if not es_video(ruta):
    raise Aviso(f"El número {VIDEO} es una imagen. Elige un video del catálogo.")
fps_original = fps_de(ruta)
frames = leer_video(ruta, desde=float(DESDE_SEG), hasta=float(HASTA_SEG))
if len(frames) < 2:
    raise Aviso("El pedazo elegido tiene menos de dos frames.")
veces = int(math.log2(MULTIPLICAR))
print(f"{corto(ruta)} · {len(frames)} frames a {fps_original:g} FPS · {(len(frames) - 1) * (MULTIPLICAR - 1)} frames nuevos")

escribir = Escritor(limpio(NOMBRE_LENTO))
anterior = cargar(frames[0])
for f in tqdm(frames[1:], desc="Cámara lenta"):
    actual = cargar(f)
    tramo(anterior, actual, veces, escribir)
    anterior = actual
escribir(anterior)
escribir.cerrar()
limpiar_temporales()
fps_salida = fps_original if RESULTADO == "más lento" else fps_original * MULTIPLICAR
montar_video(limpio(NOMBRE_LENTO), fps=fps_salida, gif=HACER_GIF_L)

In [ ]:
#@title 6.1 (Opcional) Volver a montar un video con otros ajustes
#@markdown Cambia FPS, calidad, dirección, ida y vuelta o GIF sin volver a generar los frames.
#@markdown Ejemplo: los mismos frames a 12 FPS dan un video del doble de largo y más lento.
NOMBRE_M = "secuencia" #@param {type:"string"}
FPS_MONTAJE = 24 #@param [8, 12, 15, 24, 25, 30, 48, 60] {type:"raw"}
AL_REVES_M = False #@param {type:"boolean"}
IDA_Y_VUELTA_M = False #@param {type:"boolean"}
CALIDAD_M = 18 #@param {type:"slider", min:14, max:30, step:1}
HACER_GIF_M = True #@param {type:"boolean"}

montar_video(limpio(NOMBRE_M), fps=FPS_MONTAJE, al_reves=AL_REVES_M, ida_y_vuelta=IDA_Y_VUELTA_M,
             calidad=CALIDAD_M, gif=HACER_GIF_M)

## 7. Descargar tus resultados

Descarga un `.zip` con los videos y GIFs de hoy. Los frames sueltos no se incluyen por defecto porque pesan mucho. Si conectaste Drive (celda 0.1), ya está todo guardado en `Taller_MUTAR/sesion4`.

In [ ]:
#@title 7. Descargar todo en un .zip
INCLUIR_FRAMES = False #@param {type:"boolean"}

tmp = "/content/_para_zip"
shutil.rmtree(tmp, ignore_errors=True)
shutil.copytree(CARPETA, tmp, ignore=None if INCLUIR_FRAMES else shutil.ignore_patterns("frames_*"))
ruta_zip = shutil.make_archive("/content/sesion4_film", "zip", tmp)
print(f"{ruta_zip} ({os.path.getsize(ruta_zip) / 1e6:.1f} MB)")
if os.path.getsize(ruta_zip) > 500e6:
    print("Es muy pesado para descargar desde el navegador: conecta Drive en la celda 0.1, ahí ya queda todo guardado.")
files.download(ruta_zip)

---
## Glosario

* **Interpolación de frames:** generar las imágenes que irían entre dos frames. No es un fundido: cada frame intermedio es una imagen nueva, con las partes desplazadas a mitad de camino.
* **Flujo óptico:** la estimación de hacia dónde se movió cada parte de la imagen entre un frame y otro. Es lo primero que calcula FILM.
* **Interpolaciones:** cuántas veces se parte el camino a la mitad. Con n interpolaciones salen 2ⁿ frames por transición.
* **FPS (frames por segundo):** cuántas imágenes se muestran por segundo. Los mismos frames a menos FPS dan un video más largo y lento.
* **Zoom infinito:** acercarse a una imagen hasta caer en la siguiente, sin cortes.
* **Cámara lenta:** más frames entre los frames de un video, para que dure más o se vea más fluido.

---
## El recorrido del taller

1. **Sesión 1 · DeepDream (2015):** una red entrenada para *reconocer* puede alucinar sobre una imagen y mostrarnos lo que ya conoce.
2. **Sesión 2 · BigGAN (2018):** una red entrenada para *generar* a partir de 1000 clases; entre ellas, híbridos que nadie dibujó.
3. **Sesión 3 · Projected GAN (2021):** habitaciones que parecen habitaciones y criaturas que casi existen; sin clases, inventamos nuestras propias perillas.
4. **Sesión 4 · FILM (2022):** una red que inventa lo que pasa *entre* imágenes, usada para conectar todo lo anterior.

Estas herramientas son anteriores a la difusión y a los grandes modelos de lenguaje. No buscaban un realismo perfecto, y justamente por eso sus errores (alucinaciones, híbridos, derretimientos) son materiales con los que se puede trabajar.

### Para seguir explorando

* Artículo: Reda y equipo, *FILM: Frame Interpolation for Large Motion* (ECCV 2022): https://arxiv.org/abs/2202.04901
* Repositorio oficial: https://github.com/google-research/frame-interpolation

*Taller Arqueología de la I.A. generativa · Máquinas de Estados Infinitos*